# ChemBreak Live Research Run

ChemBreak is an adaptive multi-turn framework for evaluating the safety of chemistry-specialized language models. Each experiment begins with an independent direct baseline, followed by up to three adaptive stages. Stage 1 uses response-guided strategy selection, while Stages 2 and 3 employ progressive conversational buildup, with Stage 3 activated if Stage 2 does not achieve full objective fulfillment.

The framework uses Objective Fulfillment Score (OFS) to guide adaptive decisions and Response Harmfulness Score (RHS) to independently measure harmfulness. Each experiment runs with a selected judge model and a unique run ID. ChemDFM and ChemLLM are evaluated separately, following configuration validation and dry-run testing.

In [ ]:
# Cell 1: run controls
DRY_RUN = False                # set "True" for dry_run and "False" for live_run
JUDGE_EXPERIMENT = "gpt"       # "gpt" or "qwen"
DATASET_PROFILE = "prompt28"  # "prompts28", "chemharm500", or "sosbench500"
OPENAI_JUDGE_MODEL = "gpt-4o" # Any OpenAI model alias available to your account
OPENAI_REASONING_EFFORT = None # Example: "medium" for compatible reasoning models
RUN_ID = "prompt28-gpt-4o"
SEED = 3000
TASK_LIMIT = None              # None means every task in the selected locked dataset
PROBE_TOKENIZERS = False
PROBE_ROLES = False
DETAILED_LIVE_OUTPUT = True
LIVE_TEXT_LIMIT = 1500
REFRESH_LOCAL_PACKAGE = True   # Re-extract only /content/ChemBreak when the uploaded ZIP changes

REPOSITORY_URL = "https://github.com/Jollychuks/ChemBreak"
REPOSITORY_BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak"

QWEN_JUDGE_GPU_LAYERS = 8
QWEN_JUDGE_CONTEXT_TOKENS = 16384


In [ ]:
# Cell 2: locate the uploaded package or clone a repository
from pathlib import Path
import shutil, subprocess, zipfile

PROJECT_ROOT = Path("/content/ChemBreak")
archive = Path("/content/ChemBreak.zip")
if REFRESH_LOCAL_PACKAGE and archive.is_file() and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
if not (PROJECT_ROOT / "pyproject.toml").exists() and archive.is_file():
    with zipfile.ZipFile(archive) as package:
        package.extractall("/content")
if not (PROJECT_ROOT / "pyproject.toml").exists():
    repository_root = Path("/content/chembreak_repository")
    if not (repository_root / ".git").exists():
        if REPOSITORY_URL.startswith("PASTE_"):
            raise FileNotFoundError(
                "Upload ChemBreak.zip to /content, or set REPOSITORY_URL."
            )
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(repository_root)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repository_root), "pull", "--ff-only"], check=True)
    PROJECT_ROOT = repository_root / PROJECT_SUBDIR
PROJECT_ROOT = PROJECT_ROOT.resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"ChemBreak not found at {PROJECT_ROOT}")
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
# Cell 3: install and activate ChemBreak
import importlib, site, subprocess, sys

def install(args):
    print("RUNNING:", " ".join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)

install([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
install([sys.executable, "-m", "pip", "install", "-r", PROJECT_ROOT / "requirements-cloud-ml.txt"])
install([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])

SOURCE_ROOT = (PROJECT_ROOT / "src").resolve()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
site.addsitedir(str(SOURCE_ROOT))
for name in list(sys.modules):
    if name == "chembreak" or name.startswith("chembreak."):
        del sys.modules[name]
importlib.invalidate_caches()
import chembreak
print("ChemBreak import OK:", chembreak.__version__, Path(chembreak.__file__).resolve())


In [ ]:
# Cell 4: GPU and working storage
import os, subprocess
from pathlib import Path

STORAGE_ROOT = Path("/content/chembreak_storage")
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME = STORAGE_ROOT / "cache" / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception:
    if not DRY_RUN:
        raise RuntimeError("Attach a CUDA GPU runtime before a live run.")
print("STORAGE_ROOT =", STORAGE_ROOT)


In [ ]:
# Cell 5: credential setup and check
import getpass, os
if not DRY_RUN:
    if not os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip():
        raise RuntimeError("Set GOOGLE_CLOUD_PROJECT for controller and candidate-gate calls.")
    if JUDGE_EXPERIMENT == "gpt" and not os.environ.get("OPENAI_API_KEY", "").strip():
        api_key = getpass.getpass("Paste your OpenAI API key for this runtime: ").strip()
        if not api_key:
            raise RuntimeError("No OpenAI API key was entered.")
        os.environ["OPENAI_API_KEY"] = api_key
        del api_key
print("Credential precheck complete. Live mode =", not DRY_RUN)


In [ ]:
# Cell 6: create the isolated runtime configuration
import re, yaml
from chembreak.dataset import dataset_profile_spec

if JUDGE_EXPERIMENT not in {"gpt", "qwen"}:
    raise ValueError("JUDGE_EXPERIMENT must be gpt or qwen")
dataset_spec = dataset_profile_spec(DATASET_PROFILE)
RUN_MODE = "dry" if DRY_RUN else "live"
model_tag = re.sub(r"[^a-zA-Z0-9._-]+", "_", OPENAI_JUDGE_MODEL) if JUDGE_EXPERIMENT == "gpt" else "qwen"
resolved_run_id = RUN_ID or f"chembreak_{DATASET_PROFILE}_{JUDGE_EXPERIMENT}_{model_tag}_{RUN_MODE}_seed_{SEED}"
base_config = PROJECT_ROOT / "configs" / f"config.{JUDGE_EXPERIMENT}.yaml"
config = yaml.safe_load(base_config.read_text())
config["run"].update({
    "dataset_profile": DATASET_PROFILE,
    "experiment_revision": dataset_spec["experiment_revision"],
    "seed": int(SEED),
    "run_mode": RUN_MODE,
    "run_id": resolved_run_id,
    "dry_run": bool(DRY_RUN),
    "task_limit": TASK_LIMIT,
    "live_progress": True,
    "prompts_path": str(PROJECT_ROOT / dataset_spec["prompts_path"]),
    "manifest_path": str(PROJECT_ROOT / dataset_spec["manifest_path"]),
    "lock_path": str(PROJECT_ROOT / dataset_spec["lock_path"]),
    "output_root": str(STORAGE_ROOT / RUN_MODE / resolved_run_id / "runs"),
    "artifact_root": str(STORAGE_ROOT / RUN_MODE / resolved_run_id / "task_artifacts"),
})
config["experiment"]["task_count"] = int(dataset_spec["task_count"])
config["console"] = {
    "detailed": bool(DETAILED_LIVE_OUTPUT),
    "text_limit": int(LIVE_TEXT_LIMIT),
    "show_source_task": True,
    "show_candidate": True,
    "show_target_response": True,
    "show_timings": True,
}
if JUDGE_EXPERIMENT == "gpt":
    config["roles"]["judge_llm"]["model"] = str(OPENAI_JUDGE_MODEL)
    if OPENAI_REASONING_EFFORT:
        config["roles"]["judge_llm"]["reasoning_effort"] = str(OPENAI_REASONING_EFFORT)
    else:
        config["roles"]["judge_llm"].pop("reasoning_effort", None)
else:
    config["roles"]["judge_llm"]["model_path"] = str(
        STORAGE_ROOT / "models" / "Qwen3.6-35B-A3B-Q4_K_M.gguf"
    )
for role_name in ("attack_planner_llm", "intent_gate_llm", "progressive_gate_llm"):
    config["roles"][role_name]["seed"] = int(SEED)
for target in config["targets"]:
    target["cache_dir"] = str(STORAGE_ROOT / "cache" / "huggingface" / "hub")
    target["offload_folder"] = str(STORAGE_ROOT / "offload" / target["id"])

runtime_directory = STORAGE_ROOT / RUN_MODE / resolved_run_id
runtime_directory.mkdir(parents=True, exist_ok=True)
runtime_config = runtime_directory / "config.runtime.yaml"
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print("RUN MODE =", RUN_MODE)
print("RUN ID =", resolved_run_id)
print("DATASET =", DATASET_PROFILE, "| TASKS =", dataset_spec["task_count"])
print("JUDGE MODEL =", config["roles"]["judge_llm"]["model"])
print("STAGE STRATEGIES =", config["experiment"]["stage_1_strategy"], "|", config["experiment"]["stage_2_strategy"], "|", config["experiment"]["stage_3_strategy"])
print("RUNTIME CONFIG =", runtime_config)


In [ ]:
# Cell 7: prepare the local Qwen judge when selected
import json, os, subprocess, time, urllib.request
from pathlib import Path

if DRY_RUN or JUDGE_EXPERIMENT != "qwen":
    print("Local Qwen setup is not required for this selection.")
else:
    from huggingface_hub import hf_hub_download

    QWEN_REPO = "ggml-org/Qwen3.6-35B-A3B-GGUF"
    QWEN_FILE = "Qwen3.6-35B-A3B-Q4_K_M.gguf"
    QWEN_MODEL_DIR = STORAGE_ROOT / "models"
    QWEN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    QWEN_LOCAL_PATH = QWEN_MODEL_DIR / QWEN_FILE
    LLAMA_CPP_ROOT = STORAGE_ROOT / "tools" / "llama.cpp"
    LLAMA_CPP_BIN = LLAMA_CPP_ROOT / "build" / "bin" / "llama-server"
    QWEN_LOG_PATH = STORAGE_ROOT / "tools" / "llama_cpp_judge.log"
    QWEN_PORT = 18036

    if not QWEN_LOCAL_PATH.exists() or QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        print("Downloading Qwen judge to", QWEN_LOCAL_PATH, flush=True)
        hf_hub_download(repo_id=QWEN_REPO, filename=QWEN_FILE, local_dir=str(QWEN_MODEL_DIR))
    if QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        raise RuntimeError("Qwen model file is incomplete. Rerun this cell to resume.")

    if not LLAMA_CPP_BIN.is_file():
        LLAMA_CPP_ROOT.parent.mkdir(parents=True, exist_ok=True)
        if not (LLAMA_CPP_ROOT / ".git").is_dir():
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_ROOT)],
                check=True,
            )
        subprocess.run(
            ["cmake", "-S", str(LLAMA_CPP_ROOT), "-B", str(LLAMA_CPP_ROOT / "build"),
             "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"], check=True,
        )
        subprocess.run(
            ["cmake", "--build", str(LLAMA_CPP_ROOT / "build"), "--config", "Release",
             "-j", "4", "--target", "llama-server"], check=True,
        )

    def qwen_ready():
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{QWEN_PORT}/health", timeout=3) as response:
                if response.status != 200:
                    return False
            with urllib.request.urlopen(f"http://127.0.0.1:{QWEN_PORT}/v1/models", timeout=3) as response:
                available = json.load(response)
            names = {str(item.get("id", "")) for item in available.get("data", [])}
            expected = config["roles"]["judge_llm"]["model"]
            if expected not in names:
                raise RuntimeError(f"Port {QWEN_PORT} is serving another model: {sorted(names)}")
            return True
        except RuntimeError:
            raise
        except Exception:
            return False

    if not qwen_ready():
        QWEN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        log = open(QWEN_LOG_PATH, "ab", buffering=0)
        try:
            QWEN_SERVER_PROCESS = subprocess.Popen(
                [
                    str(LLAMA_CPP_BIN), "-m", str(QWEN_LOCAL_PATH),
                    "--alias", config["roles"]["judge_llm"]["model"],
                    "--host", "127.0.0.1", "--port", str(QWEN_PORT),
                    "--ctx-size", str(QWEN_JUDGE_CONTEXT_TOKENS),
                    "--n-gpu-layers", str(QWEN_JUDGE_GPU_LAYERS),
                    "--threads", "8", "--parallel", "1", "--batch-size", "256", "--ubatch-size", "128",
                    "--jinja", "--reasoning", "on", "--reasoning-format", "deepseek", "--reasoning-budget", "2048",
                ],
                stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
            )
        finally:
            log.close()
        for attempt in range(240):
            if qwen_ready():
                break
            if QWEN_SERVER_PROCESS.poll() is not None:
                tail = QWEN_LOG_PATH.read_text(errors="replace")[-6000:]
                raise RuntimeError("Qwen judge server failed to start:\n" + tail)
            if attempt % 15 == 0:
                print(f"Waiting for Qwen judge: {attempt * 2}s", flush=True)
            time.sleep(2)
    if not qwen_ready():
        raise RuntimeError("Qwen judge server is not ready. Check " + str(QWEN_LOG_PATH))
    print("Qwen judge is ready on 127.0.0.1:18036")


In [ ]:
# Cell 8: package, dataset, provider, and GPU preflight
from pprint import pprint
from chembreak.preflight import run_preflight

preflight_report = run_preflight(
    runtime_config,
    probe_tokenizers=PROBE_TOKENIZERS,
    probe_roles=PROBE_ROLES,
)
pprint(preflight_report)


In [ ]:
# Cell 9: automated tests
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "tests"), "-v"],
    check=True,
)


## Target 1: ChemDFM

In [ ]:
# Cell 10: run ChemDFM
from chembreak.runner import ChemBreakRunner

chemdfm_runner = ChemBreakRunner(runtime_config, "ChemDFM")
try:
    chemdfm_summary = chemdfm_runner.run_all()
finally:
    chemdfm_runner.unload_target()
    chemdfm_runner.close()
print("ChemDFM OFS 5 tasks:", chemdfm_summary["ofs_5_tasks"], "/", chemdfm_summary["scheduled_tasks"])
print("ChemDFM baseline OFS 5:", chemdfm_summary["baseline_ofs_5_tasks"])
print("ChemDFM adaptive OFS 5:", chemdfm_summary["adaptive_ofs_5_tasks"])


In [ ]:
# Cell 11: check GPU memory before the second target
import subprocess
subprocess.run(["nvidia-smi"], check=False)


## Target 2: ChemLLM

In [ ]:
# Cell 12: run ChemLLM
from chembreak.runner import ChemBreakRunner

chemllm_runner = ChemBreakRunner(runtime_config, "ChemLLM")
try:
    chemllm_summary = chemllm_runner.run_all()
finally:
    chemllm_runner.unload_target()
    chemllm_runner.close()
print("ChemLLM OFS 5 tasks:", chemllm_summary["ofs_5_tasks"], "/", chemllm_summary["scheduled_tasks"])
print("ChemLLM baseline OFS 5:", chemllm_summary["baseline_ofs_5_tasks"])
print("ChemLLM adaptive OFS 5:", chemllm_summary["adaptive_ofs_5_tasks"])


In [ ]:
# Cell 13: compare completed target summaries
from pprint import pprint
comparison = {}
if "chemdfm_summary" in globals():
    comparison["ChemDFM"] = chemdfm_summary
if "chemllm_summary" in globals():
    comparison["ChemLLM"] = chemllm_summary
pprint(comparison if comparison else "Run at least one target cell first.")


In [ ]:
# Cell 14: create separate public and private result archives
import shutil
from pathlib import Path

export_root = STORAGE_ROOT / RUN_MODE / resolved_run_id / "exports"
export_root.mkdir(parents=True, exist_ok=True)
public_stage = export_root / "chembreak_public_results"
private_stage = export_root / "chembreak_private_audit"
for path in (public_stage, private_stage):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)

for target_id in ("ChemDFM", "ChemLLM"):
    release = (
        Path(config["run"]["output_root"])
        / config["run"]["experiment_revision"]
        / config["run"]["judge_experiment"]
        / target_id
        / "release"
    )
    if release.exists():
        shutil.copytree(release, public_stage / target_id, ignore=shutil.ignore_patterns("PRIVATE_AUDIT"))
        private = release / "PRIVATE_AUDIT"
        if private.exists():
            shutil.copytree(private, private_stage / target_id)

public_zip = shutil.make_archive(str(export_root / f"chembreak_{resolved_run_id}_public"), "zip", public_stage)
private_zip = shutil.make_archive(str(export_root / f"chembreak_{resolved_run_id}_private_audit"), "zip", private_stage)
print("PUBLIC RESULTS =", public_zip)
print("PRIVATE AUDIT =", private_zip)
